# Five caption-aware baselines for MOOCCubeX

This notebook replaces the earlier five baselines with five different neural recommenders:

1. **Caption-GRU4Rec** — recurrent sequential recommendation.
2. **Caption-Caser** — horizontal and vertical convolution over viewing history.
3. **Caption-NextItNet** — causal dilated residual convolutions.
4. **Caption-STAMP** — short-term attention/memory priority.
5. **Caption-MIND** — multiple learner-interest capsules using dynamic routing.

Every model receives a shared video representation built from video ID, complete caption text, concepts, course membership, and metadata. Historical tokens additionally receive behaviour, time-gap, and position information. Captions are encoded once with a multilingual Sentence Transformer and cached in Drive.

All models use identical chronological splits, full-catalog validation/test ranking, a maximum of 25 epochs, validation NDCG@10 early stopping, and independently saved outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install sentence-transformers ijson pyarrow pandas tqdm matplotlib

In [ ]:
from pathlib import Path
from dataclasses import dataclass,asdict
from collections import defaultdict
import copy,gc,json,math,random,time
import ijson,numpy as np,pandas as pd,matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from sentence_transformers import SentenceTransformer

@dataclass
class Config:
 root:str='/content/drive/MyDrive/DataCon'
 seed:int=42
 max_len:int=50
 max_concepts:int=12
 hidden:int=128
 dropout:float=.10
 negatives:int=50
 batch_size:int=128
 eval_batch_size:int=128
 max_epochs:int=25
 min_epochs:int=10
 patience:int=5
 min_delta:float=1e-4
 lr:float=1e-3
 weight_decay:float=1e-5
 grad_clip:float=5.0
 ks:tuple=(5,10,20)
 caption_model:str='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
 caption_batch_size:int=128
 max_caption_characters:int=6000
 workers:int=2
CFG=Config();ROOT=Path(CFG.root);RAW=ROOT/'MOOCCubeX';P=ROOT/'processed';S=P/'splits';G=P/'graph'
OUT=ROOT/'caption_baselines';CACHE=OUT/'cache';CKPT=OUT/'checkpoints';REPORTS=OUT/'reports'
for p in [OUT,CACHE,CKPT,REPORTS]:p.mkdir(parents=True,exist_ok=True)
def seed_all(s):
 random.seed(s);np.random.seed(s);torch.manual_seed(s)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(s)
seed_all(CFG.seed);device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device,torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print(json.dumps(asdict(CFG),indent=2))

## 1. Load chronological train, validation, and test splits

In [ ]:
needed=[S/'train.parquet',S/'valid.parquet',S/'test.parquet',G/'video_index.parquet',G/'video_metadata.parquet',
 G/'concept_video_edges.parquet',G/'course_video_edges.parquet',RAW/'entities/video.json']
missing=[str(x) for x in needed if not x.exists()]
if missing:raise FileNotFoundError(missing)
train_df=pd.read_parquet(S/'train.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)
valid_df=pd.read_parquet(S/'valid.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)
test_df=pd.read_parquet(S/'test.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)
for f in [train_df,valid_df,test_df]:
 f['user_id']=f.user_id.astype(str);f['video_id']=f.video_id.astype(str);f['timestamp']=pd.to_numeric(f.timestamp,errors='coerce').fillna(0).astype('int64')
items=sorted(train_df.video_id.unique());users=sorted(set(train_df.user_id)|set(valid_df.user_id)|set(test_df.user_id))
item2idx={x:i+1 for i,x in enumerate(items)};idx2item={i:x for x,i in item2idx.items()};user2idx={x:i for i,x in enumerate(users)}
N_ITEMS,N_USERS=len(items),len(users)
def mapped(f):
 x=f[f.video_id.isin(item2idx)].copy().reset_index(drop=True);x['u']=x.user_id.map(user2idx).astype('int64');x['i']=x.video_id.map(item2idx).astype('int64');return x
train,valid,test=map(mapped,[train_df,valid_df,test_df])
train_max=train.groupby('u').timestamp.max();v=valid.set_index('u');t=test.set_index('u');common=sorted(set(train_max.index)&set(v.index)&set(t.index))
checks={'train_before_validation':bool((train_max.loc[common].values<=v.loc[common].timestamp.values).all()),
 'validation_before_test':bool((v.loc[common].timestamp.values<=t.loc[common].timestamp.values).all()),
 'validation_catalog_known':bool(valid.i.isin(set(train.i)).all()),'test_catalog_known':bool(test.i.isin(set(train.i)).all())}
display(pd.DataFrame(checks.items(),columns=['check','passed']));assert all(checks.values())
display(pd.DataFrame([['train',len(train),train.u.nunique(),train.i.nunique()],['validation',len(valid),valid.u.nunique(),valid.i.nunique()],
 ['test',len(test),test.u.nunique(),test.i.nunique()]],columns=['split','interactions','users','videos']))

## 2. Extract complete captions and create cached multilingual embeddings

MOOCCubeX stores caption sentences in the video entity's `text` list and identifies the record with `ccid`. The cell streams the JSON, retains only the 4,149 modelling videos, joins their caption sentences, encodes them once, and saves the result. Reruns reuse the cache.

In [ ]:
video_index=pd.read_parquet(G/'video_index.parquet');video_index['video_id']=video_index.video_id.astype(str);video_index['ccid']=video_index.ccid.astype(str)
video_to_ccid=dict(zip(video_index.video_id,video_index.ccid));wanted={video_to_ccid[x] for x in items if x in video_to_ccid};ccid_to_video={video_to_ccid[x]:x for x in items if x in video_to_ccid}
CAPTION_TABLE=CACHE/'video_captions.parquet';CAPTION_EMB=CACHE/'caption_embeddings.npy'

def first_byte(path):
 with open(path,'rb') as f:
  while True:
   b=f.read(1)
   if not b or not b.isspace():return b
def stream_json(path):
 b=first_byte(path)
 if b==b'[':
  with open(path,'rb') as f:yield from ijson.items(f,'item')
 else:
  with open(path,encoding='utf-8') as f:
   for line in f:
    if line.strip():yield json.loads(line)

if not CAPTION_TABLE.exists():
 rows=[]
 for obj in tqdm(stream_json(RAW/'entities/video.json'),desc='Streaming video captions'):
  ccid=str(obj.get('ccid') or '')
  if ccid not in wanted:continue
  parts=obj.get('text') or []
  if not isinstance(parts,list):parts=[parts]
  caption=' '.join(str(x).strip() for x in parts if str(x).strip())[:CFG.max_caption_characters]
  rows.append({'video_id':ccid_to_video[ccid],'ccid':ccid,'name':str(obj.get('name') or ''),'caption':caption})
 pd.DataFrame(rows).drop_duplicates('video_id').to_parquet(CAPTION_TABLE,index=False)
captions=pd.read_parquet(CAPTION_TABLE).set_index('video_id').reindex(items)
caption_text=(captions.caption.fillna('')+' '+captions.name.fillna('')).str.strip().tolist()
if not CAPTION_EMB.exists():
 encoder=SentenceTransformer(CFG.caption_model,device=str(device))
 emb=encoder.encode(caption_text,batch_size=CFG.caption_batch_size,show_progress_bar=True,normalize_embeddings=True)
 np.save(CAPTION_EMB,emb.astype('float32'));del encoder;gc.collect();torch.cuda.empty_cache()
caption_emb=np.load(CAPTION_EMB).astype('float32');caption_matrix=np.zeros((N_ITEMS+1,caption_emb.shape[1]),dtype='float32');caption_matrix[1:]=caption_emb
print('Caption embedding shape:',caption_emb.shape,'non-empty captions:',sum(bool(x) for x in caption_text))

## 3. Prepare concepts, courses, metadata, behaviour, and histories

All continuous-feature normalization is fitted on training data only.

In [ ]:
vi_ccid={str(r.video_id):str(r.ccid) for r in video_index.itertuples()};ccid_item={vi_ccid[x]:item2idx[x] for x in items if x in vi_ccid}
cv=pd.read_parquet(G/'concept_video_edges.parquet');cv['concept_id']=cv.concept_id.astype(str);cv['ccid']=cv.ccid.astype(str);cv=cv[cv.ccid.isin(ccid_item)].drop_duplicates()
concepts=sorted(cv.concept_id.unique());c2i={x:i+1 for i,x in enumerate(concepts)};item_concepts=np.zeros((N_ITEMS+1,CFG.max_concepts),dtype='int64')
for ccid,g in cv.groupby('ccid'):
 ids=[c2i[x] for x in g.concept_id.iloc[:CFG.max_concepts]];item_concepts[ccid_item[ccid],:len(ids)]=ids
course=pd.read_parquet(G/'course_video_edges.parquet');course['video_id']=course.video_id.astype(str);course['course_id']=course.course_id.astype(str);course=course[course.video_id.isin(item2idx)].drop_duplicates('video_id')
courses=sorted(course.course_id.unique());co2i={x:i+1 for i,x in enumerate(courses)};item_course=np.zeros(N_ITEMS+1,dtype='int64')
for r in course.itertuples():item_course[item2idx[r.video_id]]=co2i[r.course_id]
meta=pd.read_parquet(G/'video_metadata.parquet');meta['video_id']=meta.video_id.astype(str);meta=meta.drop_duplicates('video_id').set_index('video_id')
MC=['duration_seconds','subtitle_sentences','subtitle_characters','concept_count'];mt=pd.DataFrame(index=items)
for c in MC:mt[c]=np.log1p(pd.to_numeric(meta.reindex(items)[c],errors='coerce').fillna(0).clip(lower=0)) if c in meta else 0
mm,ms=mt.mean(),mt.std().replace(0,1).fillna(1);mt=((mt-mm)/ms).astype('float32');item_meta=np.zeros((N_ITEMS+1,len(MC)),dtype='float32');item_meta[1:]=mt.values
BC=['watched_seconds','playback_seconds','duration_seconds','completion_ratio','segment_count','engagement_weight'];LOG={'watched_seconds','playback_seconds','duration_seconds','segment_count'}
def rawb(f):
 x=f[BC].astype('float32').replace([np.inf,-np.inf],np.nan).fillna(0).copy()
 for c in LOG:x[c]=np.log1p(x[c].clip(lower=0))
 return x
bm,bs=rawb(train).mean(),rawb(train).std().replace(0,1).fillna(1)
def normb(f):return ((rawb(f)-bm)/bs).astype('float32').values
tb,vb,xb=normb(train),normb(valid),normb(test)
def histories(f,b):
 out={}
 for u,ids in f.groupby('u',sort=False).groups.items():
  ix=np.asarray(list(ids));g=f.loc[ix];out[int(u)]={'items':g.i.astype(int).tolist(),'times':g.timestamp.astype('int64').tolist(),'beh':b[ix].tolist()}
 return out
th,vh0,xh0=histories(train,tb),histories(valid,vb),histories(test,xb);eval_users=sorted(set(th)&set(vh0)&set(xh0))
vtarget={u:vh0[u]['items'][0] for u in eval_users};xtarget={u:xh0[u]['items'][0] for u in eval_users};vh={u:copy.deepcopy(th[u]) for u in eval_users};xh={}
for u in eval_users:
 h=copy.deepcopy(th[u]);h['items'].append(vh0[u]['items'][0]);h['times'].append(vh0[u]['times'][0]);h['beh'].append(vh0[u]['beh'][0]);xh[u]=h
known={u:set(th[u]['items'])|({vtarget[u],xtarget[u]} if u in vtarget else set()) for u in th}
assert all(vtarget[u] not in vh[u]['items'] and xtarget[u] not in xh[u]['items'] for u in eval_users)
print({'concepts':len(concepts),'courses':len(courses),'evaluation_users':len(eval_users),'caption_dim':caption_emb.shape[1]})

## 4. Prefix dataset and full-catalog metrics

In [ ]:
def pad(x,n,value):x=list(x)[-n:];return [value]*(n-len(x))+x
class Prefixes(Dataset):
 def __init__(self,h):self.h=h;self.rows=[(u,j) for u,z in h.items() for j in range(1,len(z['items']))]
 def __len__(self):return len(self.rows)
 def __getitem__(self,k):
  u,j=self.rows[k];h=self.h[u];return torch.tensor(u),torch.tensor(pad(h['items'][:j],CFG.max_len,0)),torch.tensor(pad(h['times'][:j],CFG.max_len,0)),torch.tensor(pad(h['beh'][:j],CFG.max_len,[0.]*len(BC)),dtype=torch.float32),torch.tensor(h['items'][j])
ds=Prefixes(th);loader=DataLoader(ds,batch_size=CFG.batch_size,shuffle=True,num_workers=CFG.workers,pin_memory=True,persistent_workers=CFG.workers>0)
def negs(us):
 out=[]
 for u in us.tolist():
  a=[]
  while len(a)<CFG.negatives:
   i=random.randint(1,N_ITEMS)
   if i not in known[int(u)]:a.append(i)
  out.append(a)
 return torch.tensor(out)
def metrics(ranks,top10):
 r=np.asarray(ranks);o={'Accuracy@1':float((r==1).mean()),'MRR':float(np.mean(1/r)),'MeanRank':float(r.mean()),'MedianRank':float(np.median(r))}
 for k in CFG.ks:
  hit=r<=k;rec=float(hit.mean());pre=rec/k;o[f'Precision@{k}']=pre;o[f'Recall@{k}']=rec;o[f'F1@{k}']=0 if rec==0 else 2*pre*rec/(pre+rec);o[f'NDCG@{k}']=float(np.mean(np.where(hit,1/np.log2(r+1),0)));o[f'MAP@{k}']=float(np.mean(np.where(hit,1/r,0)))
 o['CatalogCoverage@10']=len(np.unique(top10))/N_ITEMS;return o
def eval_tensors(h,us):return (torch.tensor([pad(h[u]['items'],CFG.max_len,0) for u in us],device=device),torch.tensor([pad(h[u]['times'],CFG.max_len,0) for u in us],device=device),torch.tensor([pad(h[u]['beh'],CFG.max_len,[0.]*len(BC)) for u in us],dtype=torch.float32,device=device))
@torch.no_grad()
def evaluate(model,h,target,label):
 model.eval();ranks=[];tops=[];loss=0;n=0
 for z in range(0,len(eval_users),CFG.eval_batch_size):
  us=eval_users[z:z+CFG.eval_batch_size];seq,ts,b=eval_tensors(h,us);scores=model.full_scores(seq,ts,b);y=torch.tensor([target[u]-1 for u in us],device=device)
  for q,u in enumerate(us):
   seen=set(h[u]['items']);seen.discard(target[u])
   if seen:scores[q,torch.tensor([i-1 for i in seen],device=device)]=torch.finfo(scores.dtype).min
  loss+=F.cross_entropy(scores,y,reduction='sum').item();n+=len(us);ys=scores[torch.arange(len(us),device=device),y]
  ranks.extend(((scores>ys[:,None]).sum(1)+1).cpu().tolist());tops.extend((scores.topk(10,1).indices+1).cpu().tolist())
 o=metrics(ranks,tops);o['Loss']=loss/n;return o

## 5. Shared caption-aware video representation

Frozen caption embeddings are projected into the trainable recommendation space. The same content representation is used by all five models, making their comparison fair.

In [ ]:
class ContentBase(nn.Module):
 def __init__(self):
  super().__init__();d=CFG.hidden
  self.id=nn.Embedding(N_ITEMS+1,d,padding_idx=0);self.concept=nn.Embedding(len(concepts)+1,d,padding_idx=0);self.course=nn.Embedding(len(courses)+1,d,padding_idx=0)
  self.cap=nn.Sequential(nn.Linear(caption_matrix.shape[1],d),nn.GELU(),nn.LayerNorm(d));self.meta=nn.Sequential(nn.Linear(len(MC),64),nn.GELU(),nn.Linear(64,d))
  self.beh=nn.Sequential(nn.Linear(len(BC),64),nn.GELU(),nn.Linear(64,d));self.time=nn.Embedding(32,d,padding_idx=0);self.pos=nn.Embedding(CFG.max_len,d);self.norm=nn.LayerNorm(d);self.drop=nn.Dropout(CFG.dropout)
  self.register_buffer('caps',torch.tensor(caption_matrix));self.register_buffer('concept_ids',torch.tensor(item_concepts));self.register_buffer('course_ids',torch.tensor(item_course));self.register_buffer('metas',torch.tensor(item_meta))
 def static(self,i):
  c=self.concept(self.concept_ids[i]);m=self.concept_ids[i].ne(0).unsqueeze(-1);cp=(c*m).sum(-2)/m.sum(-2).clamp_min(1)
  z=self.norm(self.id(i)+self.cap(self.caps[i])+cp+self.course(self.course_ids[i])+self.meta(self.metas[i]))
  return z*i.ne(0).unsqueeze(-1)
 def tokens(self,seq,ts,b):
  gap=torch.zeros_like(ts);ok=(ts[:,1:]>0)&(ts[:,:-1]>0);gap[:,1:]=torch.where(ok,(ts[:,1:]-ts[:,:-1]).clamp_min(0),0);bucket=(torch.log2(gap.float()+1).floor().long()+1).clamp(0,31).masked_fill(ts.eq(0),0)
  p=torch.arange(CFG.max_len,device=seq.device)[None];z=self.drop(self.norm(self.static(seq)+self.beh(b)+self.time(bucket)+self.pos(p)))
  return z*seq.ne(0).unsqueeze(-1)
 def dot(self,user,cand):
  z=self.static(cand)
  if user.ndim==3:return (user.unsqueeze(2)*z.unsqueeze(1)).sum(-1).max(1).values/math.sqrt(CFG.hidden)
  return (user[:,None,:]*z).sum(-1)/math.sqrt(CFG.hidden)
 def full_scores(self,seq,ts,b):
  u=self.encode(seq,ts,b);allz=self.static(torch.arange(1,N_ITEMS+1,device=seq.device))
  if u.ndim==3:return torch.einsum('bkd,nd->bkn',u,allz).max(1).values/math.sqrt(CFG.hidden)
  return u@allz.T/math.sqrt(CFG.hidden)

## 6. The five alternative model architectures

In [ ]:
class CaptionGRU4Rec(ContentBase):
 def __init__(self):super().__init__();self.gru=nn.GRU(CFG.hidden,CFG.hidden,2,batch_first=True,dropout=CFG.dropout);self.out=nn.LayerNorm(CFG.hidden)
 def encode(self,s,t,b):return self.out(self.gru(self.tokens(s,t,b))[0][:,-1])

class CaptionCaser(ContentBase):
 def __init__(self):
  super().__init__();d=CFG.hidden;self.horizontal=nn.ModuleList([nn.Conv2d(1,32,(k,d)) for k in [1,2,3,4]]);self.vertical=nn.Conv2d(1,8,(CFG.max_len,1));self.fc=nn.Linear(32*4+8*d,d)
 def encode(self,s,t,b):
  x=self.tokens(s,t,b).unsqueeze(1);h=[F.max_pool1d(F.relu(c(x)).squeeze(3),CFG.max_len-k+1).squeeze(2) for c,k in zip(self.horizontal,[1,2,3,4])];v=F.relu(self.vertical(x)).flatten(1);return self.fc(torch.cat(h+[v],1))

class ResidualDilated(nn.Module):
 def __init__(self,d,dilation):super().__init__();self.c1=nn.Conv1d(d,d,3,dilation=dilation);self.c2=nn.Conv1d(d,d,3,dilation=2*dilation);self.d=dilation
 def causal(self,c,x,d):return c(F.pad(x,(2*d,0)))
 def forward(self,x):y=F.gelu(self.causal(self.c1,x,self.d));y=self.causal(self.c2,y,2*self.d);return F.layer_norm((x+y).transpose(1,2),(x.size(1),)).transpose(1,2)
class CaptionNextItNet(ContentBase):
 def __init__(self):super().__init__();self.blocks=nn.ModuleList([ResidualDilated(CFG.hidden,d) for d in [1,2,4,8]]);self.out=nn.LayerNorm(CFG.hidden)
 def encode(self,s,t,b):
  x=self.tokens(s,t,b).transpose(1,2)
  for block in self.blocks:x=block(x)
  return self.out(x[:,:,-1])

class CaptionSTAMP(ContentBase):
 def __init__(self):super().__init__();d=CFG.hidden;self.a=nn.Linear(d,d);self.b=nn.Linear(d,d);self.c=nn.Linear(d,d);self.w=nn.Linear(d,1);self.out=nn.Linear(2*d,d)
 def encode(self,s,t,b):
  x=self.tokens(s,t,b);mask=s.ne(0);mean=(x*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1);last=x[:,-1];a=self.w(torch.sigmoid(self.a(x)+self.b(last)[:,None]+self.c(mean)[:,None])).squeeze(-1).masked_fill(~mask,torch.finfo(x.dtype).min);ctx=(torch.softmax(a,1).unsqueeze(-1)*x).sum(1);return self.out(torch.cat([ctx,last],1))

class CaptionMIND(ContentBase):
 def __init__(self,k=4,iters=3):super().__init__();self.k=k;self.iters=iters;self.proj=nn.Linear(CFG.hidden,CFG.hidden,bias=False)
 def squash(self,x):n=x.square().sum(-1,keepdim=True);return n/(1+n)*x/torch.sqrt(n+1e-8)
 def encode(self,s,t,b):
  x=self.proj(self.tokens(s,t,b));mask=s.ne(0);logits=torch.zeros(len(s),CFG.max_len,self.k,device=s.device,dtype=x.dtype)
  for q in range(self.iters):
   w=torch.softmax(logits,2)*mask.unsqueeze(-1);caps=self.squash(torch.einsum('blk,bld->bkd',w,x))
   if q+1<self.iters:logits=logits+torch.einsum('bld,bkd->blk',x,caps)
  return caps

## 7. Shared 25-epoch trainer with early stopping

Every epoch records train loss and full-catalog validation metrics. The best checkpoint is selected by validation NDCG@10. Test data is evaluated only after training.

In [ ]:
def train_model(model,name):
 seed_all(CFG.seed);model=model.to(device);opt=torch.optim.AdamW(model.parameters(),lr=CFG.lr,weight_decay=CFG.weight_decay);scale=torch.amp.GradScaler('cuda',enabled=device.type=='cuda');best=-1;bad=0;hist=[];path=CKPT/f'{name}_best.pt'
 for epoch in range(1,CFG.max_epochs+1):
  model.train();total=0;n=0;start=time.time()
  for us,s,t,b,pos in tqdm(loader,desc=f'{name} {epoch:02d}/{CFG.max_epochs}',leave=False):
   us,s,t,b,pos=[x.to(device,non_blocking=True) for x in [us,s,t,b,pos]];negative=negs(us.cpu()).to(device);cand=torch.cat([pos[:,None],negative],1);opt.zero_grad(set_to_none=True)
   with torch.amp.autocast(device_type=device.type,enabled=device.type=='cuda'):
    u=model.encode(s,t,b);logits=model.dot(u,cand);loss=F.cross_entropy(logits,torch.zeros(len(s),dtype=torch.long,device=device))
   scale.scale(loss).backward();scale.unscale_(opt);nn.utils.clip_grad_norm_(model.parameters(),CFG.grad_clip);scale.step(opt);scale.update();total+=loss.item()*len(s);n+=len(s)
  val=evaluate(model,vh,vtarget,'validation');row={'Epoch':epoch,'TrainLoss':total/n,**{f'Val_{k}':v for k,v in val.items()},'Seconds':time.time()-start};hist.append(row);pd.DataFrame(hist).to_csv(REPORTS/f'{name}_epochs.csv',index=False)
  print(f"{name} epoch {epoch:02d}: train={row['TrainLoss']:.4f} valid={row['Val_Loss']:.4f} Recall@10={row['Val_Recall@10']:.4f} NDCG@10={row['Val_NDCG@10']:.4f} MRR={row['Val_MRR']:.4f}")
  if val['NDCG@10']>best+CFG.min_delta:best=val['NDCG@10'];bad=0;torch.save({'state':model.state_dict(),'epoch':epoch,'validation':val,'config':asdict(CFG)},path)
  else:bad+=1
  if epoch>=CFG.min_epochs and bad>=CFG.patience:print('Early stopped at',epoch);break
 saved=torch.load(path,map_location=device);model.load_state_dict(saved['state']);testm=evaluate(model,xh,xtarget,'test');row={'Model':name,'BestEpoch':saved['epoch'],'EpochsCompleted':len(hist),'BestValidationNDCG@10':saved['validation']['NDCG@10'],**{f'Test_{k}':v for k,v in testm.items()}}
 rf=REPORTS/'caption_baseline_results.csv';old=pd.read_csv(rf) if rf.exists() else pd.DataFrame();old=old[old.Model!=name] if len(old) else old;pd.concat([old,pd.DataFrame([row])],ignore_index=True).to_csv(rf,index=False);display(pd.DataFrame([row]));return hist

## 8. Train models one by one

Run one cell, wait for its saved test row, clear GPU memory, and then run the next model.

### Model 1 — Caption-GRU4Rec

In [ ]:
hist_gru=train_model(CaptionGRU4Rec(),'Caption-GRU4Rec')
gc.collect();torch.cuda.empty_cache()

### Model 2 — Caption-Caser

In [ ]:
hist_caser=train_model(CaptionCaser(),'Caption-Caser')
gc.collect();torch.cuda.empty_cache()

### Model 3 — Caption-NextItNet

In [ ]:
hist_next=train_model(CaptionNextItNet(),'Caption-NextItNet')
gc.collect();torch.cuda.empty_cache()

### Model 4 — Caption-STAMP

In [ ]:
hist_stamp=train_model(CaptionSTAMP(),'Caption-STAMP')
gc.collect();torch.cuda.empty_cache()

### Model 5 — Caption-MIND

In [ ]:
hist_mind=train_model(CaptionMIND(),'Caption-MIND')
gc.collect();torch.cuda.empty_cache()

## 9. Final comparison and saved outputs

In [ ]:
rf=REPORTS/'caption_baseline_results.csv';results=pd.read_csv(rf).sort_values('Test_NDCG@10',ascending=False).reset_index(drop=True)
numeric=[c for c in results if c not in ['Model','EpochsCompleted','BestEpoch']];display(results.style.format({c:'{:.4f}' for c in numeric}).highlight_max(subset=numeric,color='#C6EFCE'))
results.to_csv(REPORTS/'caption_baseline_results_final.csv',index=False)
ax=results.set_index('Model')[['Test_Recall@10','Test_NDCG@10','Test_MRR']].plot.bar(figsize=(11,5),rot=15);ax.grid(axis='y',alpha=.25);ax.set_title('Caption-aware baseline comparison');plt.tight_layout();plt.savefig(REPORTS/'caption_baseline_comparison.png',dpi=180,bbox_inches='tight');plt.show()
json.dump({'config':asdict(CFG),'models':results.Model.tolist(),'caption_cache':str(CAPTION_EMB),'results':str(REPORTS/'caption_baseline_results_final.csv')},open(REPORTS/'manifest.json','w'),indent=2)
print('All outputs:',OUT)

## What to do after the baselines

Compare these five caption-aware baselines against BCE-SASRec using the same test NDCG@10, Recall@10, and MRR. Run a caption ablation by replacing `caption_matrix` with zeros; the difference measures the value of caption content. Keep BCE-SASRec as the proposed model only if it improves recommendation or explanation quality under the same leakage-free protocol.